In [17]:
import os
import csv
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader


CSV_PATH = "dataset_split.csv"  
SPLIT_TRAIN = "train"
SPLIT_VAL   = "val"
SPLIT_TEST  = "test"

ORIGINAL_SHAPE = (256, 256)    
DOWNSAMPLE_TO  = (64, 64)        # Downsample frames to this size (width, height)

FRAMES_PER_VIDEO = 120           

# Model / Training settings
INPUT_DIM   = DOWNSAMPLE_TO[0] * DOWNSAMPLE_TO[1] * 3   # e.g., 64*64*3 = 12288
HIDDEN_DIM  = 256
NUM_EPOCHS  = 20
BATCH_SIZE  = 4
LEARNING_RATE = 1e-4


class RawVideoDataset(Dataset):

    def __init__(self, csv_file, split="train", resize_shape=(64, 64)):
        """
        Args:
            csv_file (str): Path to the CSV that has columns: 'filepath', 'label', 'split'.
            split (str): Which split to load ('train', 'val', or 'test').
            resize_shape (tuple): (width, height) to resize each frame.
        """
        self.samples = []
        self.resize_shape = resize_shape
        
        with open(csv_file, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    self.samples.append((row["filepath"], row["label"]))
        
        unique_labels = sorted(list(set([s[1] for s in self.samples])))
        self.label_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
        

        self.samples = [(fp, self.label_to_idx[lab]) for (fp, lab) in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Returns:
            frames_tensor: shape (120, input_dim), where input_dim = width*height*3
            label_idx: int, the label of this sequence
        """
        npy_path, label_idx = self.samples[idx]
        
        frames = np.load(npy_path) 
        
        # Downsample + flatten each frame
        processed_frames = []
        for frame in frames:
            resized_frame = cv2.resize(frame, self.resize_shape)  # shape: (64, 64, 3)
            
            
            # Flatten to 1D
            flat_frame = resized_frame.reshape(-1)  # shape: (64*64*3,)
            processed_frames.append(flat_frame)
        
        # Stack into shape (120, input_dim)
        processed_frames = np.array(processed_frames, dtype=np.float32)
        
        # Convert to torch tensor
        frames_tensor = torch.from_numpy(processed_frames)  # shape = (120, input_dim)
        
        return frames_tensor, label_idx



class VideoLSTM(nn.Module):
    """
    A simple LSTM model that takes (batch, seq_len=120, input_dim)
    and outputs a classification over the gesture label.
    """
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(VideoLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            num_layers=2,  # Stacked LSTMs
                            dropout=0.3,  # Dropout for regularization
                            bidirectional=True,  # Use bidirectional LSTM
                            batch_first=True)
        self.fc   = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x):
        """
        x.shape = (batch, 120, input_dim)
        """
        out, (h_n, c_n) = self.lstm(x)  
        
        last_out = out[:, -1, :]      
        
        logits = self.fc(last_out)     
        return logits


In [18]:

def train_direct_lstm():
    train_dataset = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_TRAIN, resize_shape=DOWNSAMPLE_TO)
    val_dataset   = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_VAL,   resize_shape=DOWNSAMPLE_TO)
    test_dataset  = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_TEST,  resize_shape=DOWNSAMPLE_TO)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
    
    num_classes = len(train_dataset.label_to_idx)
    
    model = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=num_classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    for epoch in range(NUM_EPOCHS):

        model.train()
        total_loss = 0.0
        for frames_batch, labels_batch in train_loader:

            frames_batch = frames_batch.to(device)
            labels_batch = labels_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(frames_batch) 
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_loader)
        

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for frames_batch, labels_batch in val_loader:
                frames_batch = frames_batch.to(device)
                labels_batch = labels_batch.to(device)
                
                outputs = model(frames_batch)
                loss = criterion(outputs, labels_batch)
                val_loss += loss.item()
                
                _, preds = torch.max(outputs, dim=1)
                correct += (preds == labels_batch).sum().item()
                total += labels_batch.size(0)
        
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_accuracy = (correct / total) if total > 0 else 0
        
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_accuracy*100:.2f}%")
    
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for frames_batch, labels_batch in test_loader:
            frames_batch = frames_batch.to(device)
            labels_batch = labels_batch.to(device)
            outputs = model(frames_batch)
            loss = criterion(outputs, labels_batch)
            test_loss += loss.item()
            
            _, preds = torch.max(outputs, dim=1)
            test_correct += (preds == labels_batch).sum().item()
            test_total += labels_batch.size(0)
    avg_test_loss = test_loss / len(test_loader) if len(test_loader) > 0 else 0
    test_accuracy = (test_correct / test_total) if test_total > 0 else 0
    print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracy*100:.2f}%")

    torch.save(model.state_dict(), "20_vid_lstm.pth")
    print("Model saved.")



In [19]:
if __name__ == "__main__":
    train_direct_lstm()


Epoch [1/20] | Train Loss: 2.9839 | Val Loss: 2.9559 | Val Acc: 10.00%
Epoch [2/20] | Train Loss: 2.9172 | Val Loss: 2.9062 | Val Acc: 8.33%
Epoch [3/20] | Train Loss: 2.8486 | Val Loss: 2.8578 | Val Acc: 13.33%
Epoch [4/20] | Train Loss: 2.7722 | Val Loss: 2.7888 | Val Acc: 18.33%
Epoch [5/20] | Train Loss: 2.6913 | Val Loss: 2.7418 | Val Acc: 20.00%
Epoch [6/20] | Train Loss: 2.6410 | Val Loss: 2.7124 | Val Acc: 16.67%
Epoch [7/20] | Train Loss: 2.6121 | Val Loss: 2.7144 | Val Acc: 18.33%
Epoch [8/20] | Train Loss: 2.5844 | Val Loss: 2.7249 | Val Acc: 18.33%
Epoch [9/20] | Train Loss: 2.5610 | Val Loss: 2.7145 | Val Acc: 21.67%
Epoch [10/20] | Train Loss: 2.5497 | Val Loss: 2.7120 | Val Acc: 18.33%
Epoch [11/20] | Train Loss: 2.5304 | Val Loss: 2.6943 | Val Acc: 20.00%
Epoch [12/20] | Train Loss: 2.5312 | Val Loss: 2.7065 | Val Acc: 20.00%
Epoch [13/20] | Train Loss: 2.5206 | Val Loss: 2.7669 | Val Acc: 15.00%
Epoch [14/20] | Train Loss: 2.5156 | Val Loss: 2.7767 | Val Acc: 18.33%
Ep

### TESTING with Unseen Dataset

In [20]:
model = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=20)
model.load_state_dict(torch.load("lstm_model_20.pth"))
model.eval()

/var/folders/js/b36xj6rs3k1c5z78dsb4p3p00000gn/T/ipykernel_67523/78450909.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("lstm_model_20

VideoLSTM(
  (lstm): LSTM(12288, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=512, out_features=20, bias=True)
)

In [21]:
import cv2
import torch
import numpy as np

def predict_video(video_npy_path, model, label_map, resize_shape=(64,64), device='cpu'):
    """
    video_npy_path: path to the .npy file of shape (120, H, W, 3)
    model: your trained LSTM model (already in eval mode)
    label_map: a dict mapping label_idx -> label_name, e.g. {0: "hello", 1: "thanks", ...}
    resize_shape: same as used in your RawVideoDataset
    device: 'cpu' or 'cuda'
    """
    # 1. Load frames
    frames = np.load(video_npy_path)  # shape (120, H, W, 3)
    
    # 2. Downsample + flatten each frame just like in your RawVideoDataset
    processed_frames = []
    for frame in frames:
        # Downsample
        small_frame = cv2.resize(frame, resize_shape)  # shape = (64,64,3)
        # Flatten
        flat_frame = small_frame.reshape(-1)  # shape = (64*64*3,)
        processed_frames.append(flat_frame)
    
    # 3. Convert to numpy array -> torch tensor
    processed_frames = np.array(processed_frames, dtype=np.float32)  # shape (120, 64*64*3)
    processed_tensor = torch.from_numpy(processed_frames).unsqueeze(0)  # shape (1, 120, input_dim)
    
    # 4. Move to device
    processed_tensor = processed_tensor.to(device)
    
    # 5. Model forward
    with torch.no_grad():
        outputs = model(processed_tensor)  # shape (1, num_classes)
    
    # 6. Prediction
    _, pred_idx = torch.max(outputs, dim=1)  # shape (1,)
    pred_idx = pred_idx.item()  # get the integer
    pred_label = label_map[pred_idx]
    
    return pred_label


In [22]:
import cv2
import torch
import numpy as np
import torch.nn.functional as F  # for softmax

def predict_video_with_distribution(video_npy_path, model, label_map, resize_shape=(64,64), device='cpu'):
    """
    video_npy_path: path to the .npy file of shape (120, H, W, 3)
    model: your trained LSTM model (already in eval mode)
    label_map: a dict mapping label_idx -> label_name, e.g. {0: "hello", 1: "thanks", ...}
    resize_shape: same as used in your RawVideoDataset
    device: 'cpu' or 'cuda'
    
    Returns:
        pred_label (str): The label corresponding to the highest probability
        prob_dict (dict): A dictionary {label_name: probability, ...} for all classes
    """
    # 1. Load frames
    frames = np.load(video_npy_path)  # shape (120, H, W, 3)
    
    # 2. Downsample + flatten each frame just like in your RawVideoDataset
    processed_frames = []
    for frame in frames:
        small_frame = cv2.resize(frame, resize_shape)  # shape (64,64,3)
        flat_frame = small_frame.reshape(-1)           # shape = (64*64*3,)
        processed_frames.append(flat_frame)
    
    # 3. Convert to numpy array -> torch tensor
    processed_frames = np.array(processed_frames, dtype=np.float32)  # shape (120, input_dim)
    processed_tensor = torch.from_numpy(processed_frames).unsqueeze(0)  # shape (1, 120, input_dim)
    
    # 4. Move to device
    processed_tensor = processed_tensor.to(device)
    
    # 5. Model forward
    with torch.no_grad():
        outputs = model(processed_tensor)  # shape (1, num_classes)
    
    # 6. Convert logits to probabilities via softmax
    # outputs: shape (1, num_classes)
    probs = F.softmax(outputs, dim=1)  # shape (1, num_classes)
    
    # 7. Determine predicted class
    #    We'll do argmax for predicted index
    _, pred_idx = torch.max(probs, dim=1)  # shape (1,)
    pred_idx = pred_idx.item()  
    pred_label = label_map[pred_idx]       # e.g. "hello" or "thanks"
    
    # 8. Build a dictionary {label_name: probability}
    #    for printing or returning. 
    prob_values = probs.squeeze().cpu().numpy()  # shape (num_classes,)
    prob_dict = {}
    for class_idx, class_prob in enumerate(prob_values):
        class_label = label_map[class_idx]
        prob_dict[class_label] = float(class_prob)  # convert to float for JSON, etc.
    
    return pred_label, prob_dict



In [38]:
model_loaded = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=20)
model_loaded.load_state_dict(torch.load("lstm_model_20.pth"))
# Reverse the mapping to get idx_to_label
def load_label_to_idx(csv_file):
    """
    Reads the CSV and creates the label_to_idx mapping.
    """
    with open(csv_file, "r") as f:
        reader = csv.DictReader(f)
        labels = set(row["label"] for row in reader)
    label_to_idx = {lbl: idx for idx, lbl in enumerate(sorted(labels))}
    return label_to_idx

# Load the mapping
label_to_idx = load_label_to_idx(CSV_PATH)

idx_to_label = {idx: lbl for lbl, idx in label_to_idx.items()}

# Make sure model is in eval mode
model.eval()

# Let's pick a sample .npy file
#sample_video_path = "processed_frames_test/teach/teach-test2.npy"
#sample_video_path = "processed_frames_test/class/class-test.npy"  # Replace with your test file path
sample_video_path = "processed_frames_test/room/room-test2.npy"  # for example


predicted_label, prob_distribution = predict_video_with_distribution(
    video_npy_path=sample_video_path,
    model=model,
    label_map=idx_to_label,
    resize_shape=DOWNSAMPLE_TO,  # or (64,64) or whatever you're using
    device='cpu'                 # or 'cuda' if available
)

print("Predicted label:", predicted_label)
print("Full distribution:")
for lbl, prob in prob_distribution.items():
    print(f"  {lbl}: {prob*100:.2f}%")


Predicted label: break
Full distribution:
  answer: 7.27%
  bicycle: 2.70%
  book: 7.41%
  break: 11.13%
  car: 7.31%
  class: 7.20%
  correct: 4.45%
  die: 8.68%
  exam: 2.02%
  how: 3.58%
  left: 3.09%
  lose: 0.60%
  model: 3.14%
  now: 1.61%
  page: 3.63%
  right: 7.11%
  room: 3.86%
  teach: 5.47%
  train: 3.58%
  walk: 6.16%


/var/folders/js/b36xj6rs3k1c5z78dsb4p3p00000gn/T/ipykernel_67523/14915955.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_loaded.load_state_dict(torch.load("lstm_m

#### Testing with Seen Dataset

In [31]:
train_dataset = RawVideoDataset("one_video_per_class.csv", split="train", resize_shape=(64, 64))
idx_to_label = {idx: label for label, idx in train_dataset.label_to_idx.items()}

# Path to the folder containing test files
test_folder_path = "one_video_per_class"

# Loop through all test files in the folder
for test_file in os.listdir(test_folder_path):
    if test_file.endswith(".npy"):  # Only process .npy files
        test_video_path = os.path.join(test_folder_path, test_file)
        
        # Predict the label and get the probability distribution
        pred_label, prob_dist = predict_video_with_distribution(
            video_npy_path=test_video_path,
            model=model_loaded,
            label_map=idx_to_label,
            resize_shape=(64, 64),
            device="cpu"
        )
        
        # Print results for this file
        print(f"Test File: {test_file}")
        print(f"Predicted Label: {pred_label}")
        print("Probabilities:")
        for lbl, p in prob_dist.items():
            print(f"  {lbl}: {p*100:.2f}%")
        print("-" * 40)  # Separator for readability

Test File: model-1.npy
Predicted Label: now
Probabilities:
  answer: 2.24%
  bicycle: 7.67%
  book: 0.32%
  break: 0.67%
  car: 4.63%
  class: 0.22%
  correct: 0.24%
  die: 1.18%
  exam: 0.20%
  how: 0.42%
  left: 8.95%
  lose: 1.14%
  model: 15.46%
  now: 24.02%
  page: 11.52%
  right: 4.37%
  room: 1.94%
  teach: 5.33%
  train: 7.68%
  walk: 1.79%
----------------------------------------
Test File: break-1.npy
Predicted Label: break
Probabilities:
  answer: 7.32%
  bicycle: 3.32%
  book: 8.07%
  break: 9.04%
  car: 5.51%
  class: 8.04%
  correct: 5.98%
  die: 8.48%
  exam: 3.23%
  how: 4.10%
  left: 2.74%
  lose: 0.75%
  model: 2.74%
  now: 1.07%
  page: 2.21%
  right: 6.90%
  room: 5.18%
  teach: 5.49%
  train: 3.36%
  walk: 6.46%
----------------------------------------
Test File: train-1.npy
Predicted Label: correct
Probabilities:
  answer: 5.18%
  bicycle: 6.16%
  book: 9.20%
  break: 5.79%
  car: 2.71%
  class: 8.45%
  correct: 9.68%
  die: 6.88%
  exam: 6.62%
  how: 5.16%
  lef